# 2. Find Texas images with a text query

**Idea:** search the same images in two ways: **A. captions + TF–IDF**
and **B. direct image embeddings** with Qwen3-VL-Embedding-2B.
We use six Texas images: before/after pairs from Bear Creek, Eldridge and
Kingwood. Both methods use **one query** and display their top matches.

Run the setup cell below, then either method or both. Method B works
without running Method A. See `README.txt` for the base installation.


In [ ]:
from texas_flood import load_texas_images, ask_image, show_images
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

images = load_texas_images([
    "TX001_2016_NAIP", "TX001_2017_NOAA",
    "TX002_2016_NAIP", "TX002_2017_NOAA",
    "TX003_2016_NAIP", "TX003_2017_NOAA",
    "TX004_2016_NAIP", "TX004_2017_NOAA",
    "TX011_2016_NAIP", "TX011_2017_NOAA",
    "TX012_2016_NAIP", "TX012_2017_NOAA",
    "TX013_2016_NAIP", "TX013_2017_NOAA",
    "TX018_2016_NAIP", "TX018_2017_NOAA",
    "TX019_2016_NAIP", "TX019_2017_NOAA",
    "TX020_2016_NAIP", "TX020_2017_NOAA",
])
query = "roads surrounded by floodwater"


## A. Caption search with TF–IDF

This makes one model call per image. Only the pixels and prompt are sent;
place names and acquisition dates are not included in the captions' input.


In [ ]:
prompt = "Describe visible water, roads, buildings and vegetation in two sentences. Say whether water appears outside normal channels; state uncertainty."
images["caption"] = [ask_image(path, prompt) for path in images["path"]]
images[["image_id", "caption"]]


## Search with one query

Turn the captions and query into word-feature vectors, then compare them.
Higher similarity means more overlapping terms, not a flood probability.


In [ ]:
vectorizer = TfidfVectorizer(ngram_range=(1, 2))
caption_vectors = vectorizer.fit_transform(images["caption"])
query_vector = vectorizer.transform([query])
images["score"] = cosine_similarity(query_vector, caption_vectors).ravel()
matches = images.query("score > 0").nlargest(5, "score")


In [ ]:
show_images(matches)
matches[["image_id", "score", "caption"]]


Inspect the matches: generated captions can be wrong,
and word overlap can miss synonyms or confuse “flooded” with “not flooded.”


## B. Embed the images directly with Qwen

[Qwen3-VL-Embedding-2B](https://huggingface.co/Qwen/Qwen3-VL-Embedding-2B)
turns image pixels and text into vectors in the same space. Here we embed
only the images and query; captions, filenames, locations and dates are
not model inputs. No caption-generation API calls are needed for this method.

**One-time setup:** run `%pip install -r requirements-embeddings.txt` in a
notebook cell, restart the kernel, then run the setup cell and this section.
The first load downloads several GB of model weights from Hugging Face.
The model runs on the notebook machine. A CUDA GPU is recommended; the
helper falls back to CPU, which is slower and needs substantial RAM.

[image_embeddings.py](image_embeddings.py) handles loading the model and
reading the pixels. It processes one image at a time and limits image size
to about 512 × 512 pixels to reduce memory use; small details may be lost.


In [ ]:
from image_embeddings import load_embedding_model, embed_images, embed_query

embedding_model = load_embedding_model()
image_vectors = embed_images(embedding_model, images["path"])


### Search with the same query

Both helpers return unit-length vectors, so a dot product (`@`) gives
cosine similarity. Higher scores mean closer representations. We show
the three closest images even when none is a good match; inspect them.


In [ ]:
#query = "roads surrounded by floodwater"
query = "roads all dry"
query = "all trees, no roads"

query_embedding = embed_query(embedding_model, query)
scores = (query_embedding @ image_vectors.T).ravel()
embedding_matches = images.assign(score=scores).nlargest(5, "score")
show_images(embedding_matches)
embedding_matches[["image_id", "score"]]


**Compare:** do the two methods return the same images? Compare their
rankings, not their raw scores: neither score is a flood probability.
Image embeddings can capture visual meaning that captions omit, but can
still miss floodwater or confuse it with ordinary water.

**Try next:** set `query = "houses surrounded by water"` in a new cell,
then rerun just the search cells.
Reuse `caption_vectors` and `image_vectors`; you only need to encode the
new query. For image-to-image search, replace `query_embedding` with
`embed_images(embedding_model, [images.iloc[0]["path"]])` and exclude the
query image itself from the results.


### Project ideas: Help people find useful evidence

1. **Neighborhood Flood Memory**
   Create a searchable community archive so residents can find historical flood images of places they know. Combine location and date filters with searches such as “water around houses” or “flooded open spaces.” Let residents contribute descriptions that preserve local knowledge.
   **Interface idea:** A map and image gallery with dates, sources, and available before/after comparisons.

2. **Evidence Finder for Community Improvement Requests**
   Help neighborhood groups find images that illustrate concerns they want to raise with local officials, such as repeated flooding near roads or buildings. Users search the collection, select relevant images, and assemble an illustrated evidence packet with dates, sources, and reviewed captions.
   **Interface idea:** A search gallery with an “add to report” button and a printable report preview.

3. **Our Community’s Flood Story**
   Help teachers, libraries, and community centers create an interactive exhibit about local flooding. Search for images illustrating themes such as neighborhoods, vegetation, roads, and changing water coverage. Use an LLM to draft accessible captions and discussion questions grounded in the selected images.
   **Interface idea:** A browsable story gallery with before/after views and questions visitors can explore.

4. **Quantify Images:** Help communities quantify the damage in the images. How many buildings were impacted? How much damage? Try to create a dynamic system that can use the images and AI models. 

**Build for the datathon:** Use image embeddings for discovery and an LLM/VLM on the GB10 to help explain selected evidence. Create a nice interface that makes finding, understanding, and sharing images straightforward.